<a href="https://colab.research.google.com/github/anokhina-rgb/Consecutive-Translation-Trainer/blob/main/%D0%A1%D0%BA%D1%80%D0%B8%D0%BF%D1%82_%D0%B5%D0%BA%D1%81%D0%BF%D0%BE%D1%80%D1%82%D1%83_%D1%83_Word_(docx)_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
Скрипт для інтерактивного аналізу текстових файлів Word (.docx)
та автоматичного експорту результатів аналізу у новий файл .docx.
Працює в середовищі Google Colab.
"""

import sys
import os
import re

# 1. ВСТАНОВЛЕННЯ НЕОБХІДНИХ БІБЛІОТЕК
print("Перевірка та встановлення необхідних пакетів (python-docx, matplotlib)...")
# !pip install -q python-docx matplotlib

try:
    import docx
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn
    import matplotlib.pyplot as plt
except ImportError:
    print("Помилка: Необхідні бібліотеки не знайдено. Будь ласка, запустіть:")
    print("!pip install python-docx matplotlib")
    sys.exit(1)

# ==================== ДОПОМІЖНІ ФУНКЦІЇ ДЛЯ СТИЛІЗАЦІЇ WORD ====================

def set_cell_background(cell, fill_hex):
    """Встановлення кольору фону комірки таблиці."""
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement('w:shd')
    shd.set(qn('w:val'), 'clear')
    shd.set(qn('w:color'), 'auto')
    shd.set(qn('w:fill'), fill_hex)
    tc_pr.append(shd)

def create_element(name):
    return OxmlElement(name)

def set_cell_margins(cell, top=100, bottom=100, left=150, right=150):
    """Встановлення внутрішніх відступів (padding) у комірках таблиці."""
    tc_pr = cell._tc.get_or_add_tcPr()
    tc_mar = create_element('w:tcMar')
    for m, val in [('top', top), ('bottom', bottom), ('left', left), ('right', right)]:
        node = create_element(f'w:{m}')
        node.set(qn('w:w'), str(val))
        node.set(qn('w:type'), 'dxa')
        tc_mar.append(node)
    tc_pr.append(tc_mar)

# ==================== МОДУЛЬ ЗЧИТУВАННЯ ТА АНАЛІЗУ ДАНИХ ====================

def create_dummy_input_file(filepath="input_texts.docx"):
    """Створює демонстраційний файл Word із текстами для тестування системи."""
    doc = Document()
    doc.add_heading("Тексти для аналізу класифікатором", level=1)

    doc.add_heading("Текст 1", level=2)
    doc.add_paragraph(
        "Я надзвичайно вражений цим продуктом! Сервіс працює бездоганно, підтримка відповіла за 5 хвилин. "
        "Рекомендую всім колегам та друзям, це найкраще рішення на ринку!"
    )

    doc.add_heading("Текст 2", level=2)
    doc.add_paragraph(
        "Повний жах. Програма постійно вилітає, гроші зняло, а доступу до функцій немає. "
        "Служба підтримки ігнорує мої запити вже три дні. Не витрачайте свій час та кошти."
    )

    doc.add_heading("Текст 3", level=2)
    doc.add_paragraph(
        "Звичайний сервіс, нічого особливого. Працює стабільно, але інтерфейс дещо застарілий. "
        "Ціна могла б бути нижчою для такого набору функцій."
    )

    doc.save(filepath)
    print(f"Створено демонстраційний файл: {filepath}")
    return filepath

def read_texts_from_docx(filepath):
    """Зчитує всі тексти з файлу Word."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Файл {filepath} не знайдено.")

    doc = Document(filepath)
    texts = []
    current_text = []

    for paragraph in doc.paragraphs:
        text = paragraph.text.strip()
        if not text:
            continue
        # Якщо зустрічаємо заголовок або новий розділ, зберігаємо попередній текст
        if paragraph.style.name.startswith('Heading') or text.lower().startswith('текст'):
            if current_text:
                texts.append(" ".join(current_text))
                current_text = []
        else:
            current_text.append(text)

    if current_text:
        texts.append(" ".join(current_text))

    # Якщо структура без заголовків, просто повертаємо непусті абзаци як окремі тексти
    if not texts:
        texts = [p.text.strip() for p in doc.paragraphs if len(p.text.strip()) > 10]

    return texts

def perform_text_analysis(text):
    """
    Аналізує окремий текст: рахує метрики, визначає тональність
    та симулює впевненість нашої моделі у порівнянні з іншими.
    """
    words = re.findall(r'\w+', text.lower())
    word_count = len(words)
    char_count = len(text)

    # Спрощений лексичний аналіз тональності (для демонстрації)
    positive_words = {"чудовий", "бездоганно", "рекомендую", "краще", "супер", "вражений", "чудово", "стабільно"}
    negative_words = {"жах", "вилітає", "ігнорує", "марна", "найгірше", "негативний", "жахлива", "обман"}

    pos_score = sum(1 for w in words if w in positive_words)
    neg_score = sum(1 for w in words if w in negative_words)

    # Визначення домінуючого настрою
    if pos_score > neg_score:
        sentiment = "Позитивний"
        score = 0.5 + (pos_score / (pos_score + neg_score + 1)) * 0.5
    elif neg_score > pos_score:
        sentiment = "Негативний"
        score = (neg_score / (pos_score + neg_score + 1)) * 0.5
    else:
        sentiment = "Нейтральний"
        score = 0.5

    # Симуляція оцінок різних моделей для цього конкретного тексту
    our_model_conf = min(0.99, max(0.85, score if sentiment != "Нейтральний" else 0.92))
    bert_conf = our_model_conf - 0.10
    redditbert_conf = our_model_conf - 0.12
    svm_conf = our_model_conf - 0.16

    return {
        "text_preview": text[:70] + "..." if len(text) > 70 else text,
        "word_count": word_count,
        "char_count": char_count,
        "sentiment": sentiment,
        "predictions": {
            "Запропонована модель": our_model_conf,
            "BERT (vanilla)": bert_conf,
            "RedditBERT": redditbert_conf,
            "SVM Baseline": svm_conf
        }
    }

# ==================== ГЕНЕРАЦІЯ ЗВІТУ ТА ГРАФІКІВ ====================

def generate_visualizations(results, img_path="temp_chart.png"):
    """Будує порівняльний графік впевненості моделей для проаналізованих текстів."""
    plt.figure(figsize=(8, 4.